# Deliverable 2 -- Extracción estructurada de eventos de calendario

Este notebook evalúa una estrategia de prompting mejorado para convertir mensajes
en lenguaje natural a un objeto JSON de calendario. La ejecución registrada usa
Qwen/Qwen3-4B-Instruct-2507 sobre Benchmark2 (`frases (1).json`), con fecha de
referencia 2026-09-01. El resultado guardado es 29/45 frases exactas (64.4%),
despues de aplicar normalizacion determinista a las salidas del modelo.

La métrica exacta exige que los ocho campos coincidan con el *ground truth*; las
métricas por campo permiten diagnosticar dónde falla el modelo.

In [1]:
import json
import re
import os
from datetime import datetime
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODELOS = {
    "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "qwen": "Qwen/Qwen3-4B-Instruct-2507",
}

MODELO_ACTUAL = "qwen"
MODELO_ID = MODELOS[MODELO_ACTUAL]
FECHA_REFERENCIA = "2026-09-01"
DIA_REFERENCIA = "martes"
SEMILLA = 42

print(f"Modelo: {MODELO_ID}")
print(f"Fecha de referencia: {DIA_REFERENCIA} {FECHA_REFERENCIA}")
print(f"Semilla configurada: {SEMILLA}")

Modelo: Qwen/Qwen3-4B-Instruct-2507
Fecha de referencia: martes 2026-09-01
Semilla configurada: 42


In [2]:
# ============================================================
# CARGA Y VALIDACION DEL BENCHMARK
# ============================================================

# Se prioriza Benchmark2; el segundo archivo permite ejecutar el notebook
# localmente o en Colab sin depender de una ruta absoluta.
BENCHMARK_CANDIDATOS = [
    os.path.join(os.getcwd(), "frases (1).json"),
    "/content/frases (1).json",
    os.path.join(os.getcwd(), "frases.json"),
    "/content/frases.json",
]
BENCHMARK_FILE = next((p for p in BENCHMARK_CANDIDATOS if os.path.exists(p)), None)
if BENCHMARK_FILE is None:
    raise FileNotFoundError(
        "No se encontro el benchmark. Sube 'frases (1).json' a Colab."
    )

with open(BENCHMARK_FILE, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

frases_benchmark = benchmark.get("frases", [])
total_declarado = benchmark.get("metadata", {}).get("total_frases")
if not frases_benchmark or total_declarado != len(frases_benchmark):
    raise ValueError("El benchmark no coincide con su total declarado.")

print(f"Benchmark: {os.path.basename(BENCHMARK_FILE)} | {len(frases_benchmark)} frases")

Benchmark: frases (1).json | 45 frases


In [3]:
# ============================================================
# CARGA DEL MODELO
# ============================================================
torch.manual_seed(SEMILLA)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEMILLA)

tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODELO_ID, torch_dtype=torch.float16, device_map="auto"
)
print(f"Modelo cargado en: {model.device}")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Modelo cargado en: cuda:0


In [4]:
# ============================================================
# SYSTEM PROMPT: Reglas + Few-Shot (CORREGIDO)
# ============================================================

SYSTEM_PROMPT = """Eres un extractor de datos de calendario.

REGLAS:
1. HOY ES {dia} {fecha}. Calcula fechas relativas desde esta fecha.
2. Si un campo NO se menciona, usa null (o [] para participantes). NUNCA inventes.
3. titulo: nombre breve del evento; elimina fecha, hora, participantes y ubicacion, pero conserva sus descriptores (p. ej., "Cita médica" o "Reunión de planificación"). Conserva el idioma de la frase.
4. fecha: YYYY-MM-DD. hora_inicio: HH:MM 24h.
5. duracion_minutos: entero (1 hora=60). Si no hay = null.
6. participantes: lista de nombres reales. Si no hay = [].
7. ubicacion: donde se realiza. Si no se menciona = null.
8. recordatorio_minutos: entero. Si no se menciona = null.
9. zona_horaria: "America/Santiago" por defecto; Madrid = "Europe/Madrid" y Argentina = "America/Argentina/Buenos_Aires".
10. Si se menciona virtual, videollamada o una plataforma, extraelo como ubicacion.
11. Si una fecha u hora es invalida, usa null en ese campo; no la corrijas inventando datos.

EJEMPLOS:

Frase: "Reunión con Ana el viernes a las 10am"
{{"titulo": "Reunión", "fecha": "2026-09-04", "hora_inicio": "10:00", "duracion_minutos": null, "participantes": ["Ana"], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "Reunión de 2 horas el 15/09 a las 9:30 en la sala de conferencias, recordar 30 min"
{{"titulo": "Reunión", "fecha": "2026-09-15", "hora_inicio": "09:30", "duracion_minutos": 120, "participantes": [], "ubicacion": "Sala de conferencias", "recordatorio_minutos": 30, "zona_horaria": "America/Santiago"}}

Frase: "Reunión"
{{"titulo": "Reunión", "fecha": null, "hora_inicio": null, "duracion_minutos": null, "participantes": [], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "Meeting on Friday at 10am"
{{"titulo": "Meeting", "fecha": "2026-09-04", "hora_inicio": "10:00", "duracion_minutos": null, "participantes": [], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "Junta virtual mañana a las 5 con el profe"
{{"titulo": "Junta", "fecha": "2026-09-02", "hora_inicio": "17:00", "duracion_minutos": null, "participantes": ["el profe"], "ubicacion": "virtual", "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}"""
def construir_prompt(frase_texto):
    # Ahora format solo reemplazará {dia} y {fecha} porque los demás tienen doble llave
    system = SYSTEM_PROMPT.format(dia=DIA_REFERENCIA, fecha=FECHA_REFERENCIA)
    user_msg = f'Frase: "{frase_texto}"\n\nRespuesta (SOLO JSON):'
    if MODELO_ACTUAL == "qwen":
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user_msg}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    else:
        return f"<|system|>\n{system}\n<|user|>\n{user_msg}\n<|assistant|>\n"

print("Prompt configurado correctamente con llaves escapadas")

Prompt configurado correctamente con llaves escapadas


In [5]:
# ============================================================
# POST-PROCESAMIENTO Y VALIDACION DE ESQUEMA
# ============================================================

CAMPOS = ["titulo", "fecha", "hora_inicio", "duracion_minutos",
          "participantes", "ubicacion", "recordatorio_minutos",
          "zona_horaria"]

def extraer_json(respuesta):
    """Extrae el primer objeto JSON valido, incluso con texto adicional."""
    if not isinstance(respuesta, str):
        return {"error": "No JSON detected"}
    decoder = json.JSONDecoder()
    for match in re.finditer(r"\{", respuesta):
        try:
            objeto, _ = decoder.raw_decode(respuesta[match.start():])
            if isinstance(objeto, dict):
                return objeto
        except json.JSONDecodeError:
            continue
    return {"error": "No JSON detected", "raw": respuesta[:200]}

def normalizar_fecha(valor):
    if not isinstance(valor, str) or not re.fullmatch(r"\d{4}-\d{2}-\d{2}", valor):
        return None
    try:
        datetime.strptime(valor, "%Y-%m-%d")
        return valor
    except ValueError:
        return None

def normalizar_hora(valor, frase_texto):
    if not isinstance(valor, str):
        return None
    m = re.fullmatch(r"\s*(\d{1,2})(?::(\d{2}))?\s*(am|pm)?\s*", valor, re.I)
    if not m:
        return None
    hora = int(m.group(1))
    minuto = int(m.group(2) or 0)
    sufijo = (m.group(3) or "").lower()
    if sufijo == "pm" and hora < 12:
        hora += 12
    elif sufijo == "am" and hora == 12:
        hora = 0
    elif not sufijo and 1 <= hora <= 7 and "de la mañana" not in frase_texto.lower():
        hora += 12
    if hora > 23 or minuto > 59:
        return None
    return f"{hora:02d}:{minuto:02d}"

def post_procesar(json_salida, frase_texto):
    if not isinstance(json_salida, dict) or "error" in json_salida:
        return json_salida

    # Mantener exactamente los ocho campos del contrato de salida.
    r = {c: json_salida.get(c, [] if c == "participantes" else None) for c in CAMPOS}
    r["fecha"] = normalizar_fecha(r.get("fecha"))
    r["hora_inicio"] = normalizar_hora(r.get("hora_inicio"), frase_texto)

    participantes = r.get("participantes")
    r["participantes"] = ([x.strip() for x in participantes
                            if isinstance(x, str) and x.strip()
                            and not x.startswith("<") and not x.strip().isdigit()]
                           if isinstance(participantes, list) else [])

    for campo in ["duracion_minutos", "recordatorio_minutos"]:
        valor = r.get(campo)
        if isinstance(valor, bool):
            r[campo] = None
        elif valor is not None and not isinstance(valor, (int, float)):
            numero = re.search(r"(\d+)", str(valor))
            r[campo] = int(numero.group(1)) if numero else None

    texto = frase_texto.lower()
    if "madrid" in texto:
        r["zona_horaria"] = "Europe/Madrid"
    elif "argentina" in texto:
        r["zona_horaria"] = "America/Argentina/Buenos_Aires"
    elif not isinstance(r.get("zona_horaria"), str) or not r["zona_horaria"].strip():
        r["zona_horaria"] = "America/Santiago"

    if not r.get("ubicacion"):
        for etiqueta in ["virtual", "videollamada", "Google Meet", "Zoom", "Teams"]:
            if etiqueta.lower() in texto:
                r["ubicacion"] = etiqueta
                break
    return r

print("Post-procesador listo: esquema, fechas, horas y zonas horarias validadas")

Post-procesador listo: esquema, fechas, horas y zonas horarias validadas


In [6]:
# ============================================================
# EJECUCION
# ============================================================
resultados = []

for frase in benchmark["frases"]:
    prompt = construir_prompt(frase["texto"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=256, temperature=0.01,
            do_sample=True, top_p=0.95, pad_token_id=tokenizer.eos_token_id
        )

    respuesta = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True).strip()
    json_salida = extraer_json(respuesta)

    json_salida = post_procesar(json_salida, frase["texto"])
    resultados.append({"id": frase["id"], "texto": frase["texto"], "resultado_modelo": json_salida})
    print(f"Frase {frase['id']:2d}: {'OK' if 'error' not in json_salida else 'ERROR'}")

output_file = f"resultados_{MODELO_ACTUAL}_mejorado.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)
print(f"\nGuardado: {output_file}")

Frase  1: OK
Frase  2: OK
Frase  3: OK
Frase  4: OK
Frase  5: OK
Frase  6: OK
Frase  7: OK
Frase  8: OK
Frase  9: OK
Frase 10: OK
Frase 11: OK
Frase 12: OK
Frase 13: OK
Frase 14: OK
Frase 15: OK
Frase 16: OK
Frase 17: OK
Frase 18: OK
Frase 19: OK
Frase 20: OK
Frase 21: OK
Frase 22: OK
Frase 23: OK
Frase 24: OK
Frase 25: OK
Frase 26: OK
Frase 27: OK
Frase 28: OK
Frase 29: OK
Frase 30: OK
Frase 31: OK
Frase 32: OK
Frase 33: OK
Frase 34: OK
Frase 35: OK
Frase 36: OK
Frase 37: OK
Frase 38: OK
Frase 39: OK
Frase 40: OK
Frase 41: OK
Frase 42: OK
Frase 43: OK
Frase 44: OK
Frase 45: OK

Guardado: resultados_qwen_mejorado.json


In [7]:
# ============================================================
# EVALUACION vs GROUND TRUTH CON REPORTE DESCARGABLE (CSV)
# ============================================================
from google.colab import files
import csv

CAMPOS = ["titulo", "fecha", "hora_inicio", "duracion_minutos",
          "participantes", "ubicacion", "recordatorio_minutos", "zona_horaria"]

correctos = 0
errores_campo = {c: 0 for c in CAMPOS}
correctos_categoria = {c: {"total": 0, "correctos": 0} for c in benchmark.get("metadata", {}).get("categorias", {})}
errores_tipo = {"inventado": 0, "omision": 0, "valor_incorrecto": 0}
detalles = []  # Aquí guardamos cada registro para el CSV

for r in resultados:
    # Buscar el ground truth correspondiente
    gt = next((f for f in benchmark["frases"] if f["id"] == r["id"]), None)
    if gt is None:
        continue
    gt = gt["ground_truth"]
    pred = r["resultado_modelo"]
    categoria = next(f.get("categoria") for f in benchmark["frases"] if f["id"] == r["id"])
    correctos_categoria.setdefault(categoria, {"total": 0, "correctos": 0})["total"] += 1

    # Si el modelo devolvió un error, marcamos todos los campos como incorrectos
    if "error" in pred:
        for c in CAMPOS:
            errores_campo[c] += 1
            errores_tipo["valor_incorrecto"] += 1
        detalles.append({
            "id": r["id"],
            "texto": r["texto"],
            "exacto": False,
            **{f"{c}_gt": gt.get(c) for c in CAMPOS},
            **{f"{c}_pred": "ERROR_MODELO" for c in CAMPOS},
            **{f"{c}_ok": False for c in CAMPOS}
        })
        continue

    ok_general = True
    row_detail = {
        "id": r["id"],
        "texto": r["texto"],
        "exacto": True
    }

    for c in CAMPOS:
        esp = gt.get(c)
        opt = pred.get(c)
        field_ok = True  # Asumimos que es correcto

        # Lógica de comparación (igual que antes)
        if esp is None and opt is None:
            pass  # Ambos nulos -> correcto
        elif esp is None and opt is not None:
            errores_campo[c] += 1
            errores_tipo["inventado"] += 1
            ok_general = False
            field_ok = False
        elif esp is not None and opt is None:
            errores_campo[c] += 1
            errores_tipo["omision"] += 1
            ok_general = False
            field_ok = False
        elif isinstance(esp, list):
            # Comparar listas (participantes) ignorando mayúsculas
            pred_list = [str(x).lower() for x in (opt or [])]
            gt_list = [str(x).lower() for x in esp]
            if set(gt_list) != set(pred_list):
                errores_campo[c] += 1
                errores_tipo["valor_incorrecto"] += 1
                ok_general = False
                field_ok = False
        else:
            # Comparación simple de strings
            if str(esp).lower() != str(opt).lower():
                errores_campo[c] += 1
                errores_tipo["valor_incorrecto"] += 1
                ok_general = False
                field_ok = False

        # Guardamos los valores para el reporte
        row_detail[f"{c}_gt"] = esp
        row_detail[f"{c}_pred"] = opt
        row_detail[f"{c}_ok"] = field_ok

    row_detail["exacto"] = ok_general
    if ok_general:
        correctos += 1
        correctos_categoria[categoria]["correctos"] += 1
    detalles.append(row_detail)

# ---- Mostrar resultados en consola ----
total = len(resultados)
print(f"\n{'='*50}")
print(f"  {MODELO_ACTUAL.upper()} MEJORADO")
print(f"  Precision: {correctos}/{total} ({correctos/total*100:.1f}%)")
print(f"{'='*50}")
for c in CAMPOS:
    ok = total - errores_campo[c]
    print(f"  {c:25s}: {ok}/{total} ({ok/total*100:.1f}%)")
print("\n  Precision por categoria:")
for categoria, valores in correctos_categoria.items():
    if valores["total"]:
        p = valores["correctos"] / valores["total"] * 100
        print(f"  {categoria:20s}: {valores['correctos']}/{valores['total']} ({p:.1f}%)")
print(f"\n  Errores de campo: {errores_tipo}")

# ---- GENERAR Y DESCARGAR REPORTE CSV ----
reporte_file = f"reporte_{MODELO_ACTUAL}.csv"

with open(reporte_file, "w", newline='', encoding='utf-8') as f:
    # Construir los encabezados dinámicamente
    base_headers = ["id", "texto", "exacto"]
    field_headers = []
    for c in CAMPOS:
        field_headers.extend([f"{c}_gt", f"{c}_pred", f"{c}_ok"])
    headers = base_headers + field_headers

    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()

    # Escribir cada detalle
    for row in detalles:
        # Asegurar que todas las claves existen (por si falta alguna)
        write_row = {h: row.get(h, "") for h in headers}
        writer.writerow(write_row)

    # ---- Añadir resumen al final del CSV ----
    writer.writerow({h: "" for h in headers})  # Línea en blanco
    writer.writerow({"id": "RESUMEN", "texto": f"Exactitud General: {correctos}/{total} ({correctos/total*100:.1f}%)"})
    for c in CAMPOS:
        ok = total - errores_campo[c]
        writer.writerow({"id": f"Campo_{c}", "texto": f"{ok}/{total} ({ok/total*100:.1f}%)"})

# Descargar el archivo a tu máquina
files.download(reporte_file)
print(f"\n✅ Reporte descargado: {reporte_file}")


  QWEN MEJORADO
  Precision: 29/45 (64.4%)
  titulo                   : 37/45 (82.2%)
  fecha                    : 37/45 (82.2%)
  hora_inicio              : 45/45 (100.0%)
  duracion_minutos         : 45/45 (100.0%)
  participantes            : 42/45 (93.3%)
  ubicacion                : 45/45 (100.0%)
  recordatorio_minutos     : 45/45 (100.0%)
  zona_horaria             : 45/45 (100.0%)

  Por categoria:
  simple                  : 7/8 (87.5%)
  con_participantes       : 5/7 (71.4%)
  ambigua                 : 3/7 (42.9%)
  compleja                : 5/7 (71.4%)
  edge_case               : 6/8 (75.0%)
  chat                    : 3/8 (37.5%)

  Errores de campo: {'inventado': 0, 'omision': 0, 'valor_incorrecto': 19}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Reporte descargado: reporte_qwen.csv
